In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch

# I'm using torch.tensor() to wrap the existing numpy arrays.
# I added .float() at the end because PyTorch weights are usually float32.
X_train_tensor = torch.tensor(X_train).float()
X_test_tensor = torch.tensor(X_test).float()

# For the targets (y), I also need them to be float because this is a regression problem
y_train_tensor = torch.tensor(y_train).float()
y_test_tensor = torch.tensor(y_test).float()

# Checking the shapes just to be safe
print(f"X_train tensor shape: {X_train_tensor.shape}")
print(f"y_train tensor shape: {y_train_tensor.shape}")

In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import TensorDataset

# I'm using TensorDataset to bundle my features (X) and targets (y) together.
# This makes it super easy to grab a pair of (image, age) just by indexing like train_dataset[0].
# It's like zipping them into a list of tuples, but optimized for PyTorch.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Verifying it worked by printing the length
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# I did this because the DataLoader needs a proper 'Dataset' object


In [ ]:
# 3. Create DataLoaders

from torch.utils.data import DataLoader



batch_size = 32

# For training, I set shuffle=True.
# This is crucial because if the data was sorted
# the model would get biased during the first few batches. Shuffling mixes it up.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# For testing, I set shuffle=False.
# I don't need to shuffle the test data because I'm just evaluating performance,
# and keeping the order the same makes debugging easier if I want to check specific errors later.
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Checking how many batches we have
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

In [ ]:
# 4. Print shape of one batch

# I'm using iter() and next() to manually grab just the very first batch of data from the loader.
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Printing the shapes
print(f"Batch images shape: {images.shape}")
print(f"Batch labels shape: {labels.shape}")

In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

# I'm creating a grid of 4 images just to see what the data actually looks like.
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

for i in range(4):
    img_to_show = images[i].permute(1, 2, 0)

    axes[i].imshow(img_to_show)
    axes[i].set_title(f"Age: {labels[i].item()}") # .item() gets the actual number from the tensor
    axes[i].axis('off') # Hiding axes ticks because they are distracting

plt.show()


In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn
import torch.nn.functional as F


class AgePredictor(nn.Module):
    def __init__(self):
        super(AgePredictor, self).__init__()

        # Layer 1: Input to Hidden 1
        self.fc1 = nn.Linear(3 * 36 * 36, 512)

        # Layer 2: Hidden 1 to Hidden 2
        # I'm reducing the size gradually (512 -> 128) to compress the information.
        self.fc2 = nn.Linear(512, 128)

        # Layer 3: Hidden 2 to Hidden 3
        self.fc3 = nn.Linear(128, 64)

        # Layer 4: Hidden 3 to Output
        self.fc4 = nn.Linear(64, 1)

    def forward(self, x):
        # Step 1: Flattening
        x = x.view(x.size(0), -1)

        # Step 2: Passing through layers with ReLU
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))

        # Step 3: Output Layer
        # I am not using an activation function (like Sigmoid or ReLU) here.
        # Because this is Regression. I want the raw number.
        # If I used Sigmoid, predictions would be stuck between 0 and 1, which is bad for age
        x = self.fc4(x)

        return x

# Instantiating the model to check it
model = AgePredictor()
print(model)

In [ ]:
# Task 2: Write your training loop here:

def train_model(model, train_loader, criterion, optimizer, num_epochs=10):

    # I'm looping through the number of epochs
    for epoch in range(num_epochs):

        # Setting the model to training mode.
        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            # 1. Zero the gradients
            # I have to do this because PyTorch accumulates gradients by default.
            # If I don't clear them, the new gradients get added to the old ones, creating a mess.
            optimizer.zero_grad()

            # 2. Forward Pass
            outputs = model(images)

            # 3. Calculate Loss
            loss = criterion(outputs, labels.view(-1, 1))

            # 4. Backward Pass (Backpropagation)
            # This calculates the gradients
            loss.backward()

            # 5. Optimizer Step
            # This updates the weights using the gradients we just calculated.
            optimizer.step()

            running_loss += loss.item()

        # Printing the average loss for this epoch
        # This helps me see if the model is actually learning (loss should go down).
        epoch_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

In [ ]:
# Task 3: Write your validation loop here:

def validate_model(model, test_loader, criterion):

    # I'm switching the model to 'eval' mode.
    # This freezes layers like Dropout or BatchNorm so they behave consistently during testing.
    model.eval()

    running_loss = 0.0
    running_mae = 0.0

    # I'm using torch.no_grad() here I don't need to calculate gradients for backpropagation during testing.
    # This saves a ton of memory and makes the code run faster.
    with torch.no_grad():
        for images, labels in test_loader:

            # Forward pass only
            outputs = model(images)

            # Calculating the Loss (MSE)
            loss = criterion(outputs, labels.view(-1, 1))
            running_loss += loss.item()

            # Calculating MAE (Mean Absolute Error) manually.
            mae = torch.abs(outputs - labels.view(-1, 1)).mean()
            running_mae += mae.item()

    # Calculating averages
    avg_loss = running_loss / len(test_loader)
    avg_mae = running_mae / len(test_loader)

    print(f"Validation Loss: {avg_loss:.4f} | Validation MAE: {avg_mae:.4f}")
    return avg_mae

In [ ]:
# Task 4: Define device, model, loss, optimizer:
import torch.optim as optim

# Task: Define device, model, loss function, and optimizer

# 1. Define Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Initialize Model
model = AgePredictor().to(device)

# 3. Define Loss Function
# I chose MSELoss (Mean Squared Error) because this is a Regression problem.
# It penalizes large errors heavily, which helps the model fix big mistakes quickly.
criterion = nn.MSELoss()

# 4. Define Optimizer
# I chose Adam with a learning rate (lr) of 0.001.
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:


train_losses = []
val_maes = []

num_epochs = 20

print("Starting Training...")

for epoch in range(num_epochs):
    # Training Phase
    model.train() # Set to train mode
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels.view(-1, 1))

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Calculating average training loss for this epoch
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    # Validation Phase
    model.eval() # Set to eval mode
    val_loss = 0.0
    val_mae = 0.0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            # Calculating MAE (Mean Absolute Error) in years
            mae = torch.abs(outputs - labels.view(-1, 1)).mean()
            val_mae += mae.item()

    avg_val_mae = val_mae / len(test_loader)
    val_maes.append(avg_val_mae)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {epoch_loss:.4f} | Val MAE: {avg_val_mae:.2f} years")

print("Training Complete!")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

# Task: Plot the training loss over epochs

plt.figure(figsize=(12, 5))

# Plotting Training Loss (MSE)
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss (MSE)', color='blue')
plt.title('Training Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss (Mean Squared Error)')
plt.grid(True)


In [ ]:
# Task: Plot the validation loss over epochs
# Plotting Validation MAE
plt.subplot(1, 2, 2)
plt.plot(val_maes, label='Validation MAE (Years)', color='orange')
plt.title('Validation MAE over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error (Years)')
plt.grid(True)

plt.show()

# The lines should go down
# If Training goes down but Validation goes up, that means I'm overfitting

In [ ]:
# Task 2 (Bonus): Write your code here:

# 1. Get a batch of test data
dataiter = iter(test_loader)
images, labels = next(dataiter)



# 3. Make predictions
model.eval() # Safety first: switch to eval mode!
with torch.no_grad():
    outputs = model(images)

# 4. Prepare for plotting
images_cpu = images.cpu()
labels_cpu = labels.cpu()
preds_cpu = outputs.cpu()

# 5. Plotting
fig, axes = plt.subplots(2, 4, figsize=(15, 8)) # A grid of 8 images (2 rows, 4 columns)
axes = axes.flatten()

for i in range(8):
    # Permuting again: (C, H, W) -> (H, W, C) for matplotlib
    img_to_show = images_cpu[i].permute(1, 2, 0)

    # Getting the numbers
    actual_age = labels_cpu[i].item()
    pred_age = preds_cpu[i].item()

    axes[i].imshow(img_to_show)

    # I'm formatting the title to show both numbers.
    # I rounded the prediction (.1f) because the model gives floats like 24.532 years.
    axes[i].set_title(f"Actual: {actual_age:.0f} | Pred: {pred_age:.1f}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()